#### Build Multi Agent With LangChain

In [63]:
# Import Packages
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage, SystemMessage, ToolMessage
from tavily import TavilyClient
from dotenv.ipython import load_dotenv
from IPython.display import Markdown
import os
from pprint import pprint

In [64]:
# Import ApIKeys
load_dotenv(".env")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [65]:
# Prepare The MIND LLMs
# basic_mind = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, max_tokens=4696, api_key=GOOGLE_API_KEY)
# advanced_mind = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0, max_tokens=4696, api_key=GOOGLE_API_KEY)
basic_groq_llm = ChatGroq(model="qwen/qwen3-32b", temperature=0, max_tokens=None, reasoning_format="parsed",timeout=None, max_retries=2)

In [66]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

#### Agent for generating marketing offres

In [67]:
# Prepare The Body - Tools

@tool
def get_client_data(name: str):
    # Search on Data Base About Client
    '''
        Get Client Information About Marketing
    '''
    data = {
        "client_id": 12548,
        "fullName": name,
        "age": 34,
        "gender": "Male",
        "city": "Casablanca",
        "phone": "+212612345678",
        "email": "ahmed.elamrani@gmail.com",

        "customer_since": "2021-03-15",
        "last_purchase_date": "2026-06-20",
        "total_orders": 18,
        "total_spent": 24500,
        "average_order_value": 1361,

        "favorite_category": "Electronics",
        "favorite_brand": "Samsung",
        "purchase_frequency": "Monthly",

        "last_products": [
            "Samsung Galaxy S25",
            "Galaxy Buds Pro",
            "Wireless Charger"
        ],

        "loyalty_points": 3200,
        "membership": "Gold",

        "preferred_channel": "WhatsApp",
        "marketing_opt_in": True
    }
    return data


@tool
def tips_for_marketing_offre(query: str):
    '''
        Return List of Best Practices for generate marketing offre
    '''
    try:
        results = tavily_client.search(query=query)
        formatted_result = []
        for i, content in enumerate(results.get("results")):
            formatted_result.append(f"url: {content.get("url")}, title: {content.get("title")}, content: {content.get("content")}")
        return {"results": "\n".join(formatted_result)}
    except:
        return {"results": "WebSearch Tool Can Not get Informations"}


In [74]:
result = tips_for_marketing_offre.func("How is abdellah karani ?")
pprint(result)

{'results': 'url: https://talent.gravityer.com/abdellah-karani, title: '
            'Abdellah Karani | Gravity Talent Portal, content: Data Scientist '
            'passionate about leveraging data-driven insights to solve '
            'real-world problems. Experienced in AI/ML model development, NLP '
            'pipelines, ...Read more\n'
            'url: https://github.com/KaraniAbdellah, title: karani Abdellah '
            "KaraniAbdellah, content: I'm Abdellah 🌱 I'm currently focused on "
            'AI, ML, Data Engineering, and Cloud 👨 💻 I build real-world '
            'projects in Python, ML pipelines, and backend systems\n'
            'url: https://en.wikipedia.org/wiki/Abdullah_Al-Karni, title: '
            'Abdullah Al-Karni, content: Abdullah Al-Karni (born 8 August '
            '1976) is a Saudi Arabian former footballer. He competed in the '
            "men's tournament at the 1996 Summer Olympics.Read more\n"
            'url: '
            'https://www.linked

In [71]:
# Create Agent
sys_prompt = """
        You are a Marketing AI Agent specialized in generating personalized marketing offers.
        When a user requests a marketing offer:
            1. First retrieve the customer information using getClientData.
            2. Analyze customer profile
            3. generate marketing offre
            4. Generate a personalized marketing offer.
"""

marketing_agent = create_agent(
    model=basic_groq_llm,
    tools=[get_client_data, tips_for_marketing_offre],
    system_prompt=sys_prompt,
    debug=True
)

In [ ]:
# Ask Agent
config = {"configurable": {"thread_id": 1}}
res = marketing_agent.invoke(
    input={"messages": [HumanMessage("Generate Marketing Offre for Abdellah Karani.")]},
    config=config
)
display(Markdown(res["messages"][-1].content))

[values] {'messages': [HumanMessage(content='Generate Marketing Offre for Abdellah Karani', additional_kwargs={}, response_metadata={}, id='05001f09-af25-4846-8ed9-091bbafa1b70')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to generate a marketing offer for Abdellah Karani. Let me start by following the steps outlined in the instructions.\n\nFirst, I need to retrieve the client\'s information using the get_client_data function. The required parameter is the name, which in this case is "Abdellah Karani". So I\'ll call that function with his name to get his details.\n\nOnce I have the client data, I\'ll analyze his profile. I need to look at his preferences, past interactions, and any other relevant information from the data retrieved. This analysis will help me understand what kind of offer would be most effective for him.\n\nNext, I should use the tips_for_marketing_offre function to get best practices for cr

**Marketing Offer for Abdellah Karani**  

**Subject:** Exclusive Samsung Electronics Deal + 500 Loyalty Points for You, Abdellah!  

Hi Abdellah,  

As a valued Gold member and a Samsung enthusiast, we’ve crafted a special offer just for you!  

**Your Offer:**  
- **15% OFF** on your next Samsung Electronics purchase (e.g., Galaxy S25 Ultra, Buds 2 Pro, or Smart Home Devices).  
- **Free Wireless Charger** with any purchase over MAD 2,000.  
- **Double Loyalty Points (1,000 points)** on this order—boosting your total to **4,200 points**!  

**Why Act Now?**  
- Your last Samsung purchase was 6 months ago—time for an upgrade!  
- This offer expires in **7 days** to ensure exclusivity.  

**How to Redeem:**  
1. Reply to this WhatsApp message with your preferred product.  
2. Use code **KARANI15** at checkout.  

We know you love shopping monthly, so let’s make your next order even better!  

Looking forward to serving you,  
[Your Company Name]  

---  
*P.S. As a Gold member, you’ll also receive priority support and early access to new Samsung launches!*  

**Preferred Contact:** WhatsApp @ +212612345678 | **Offer Valid Until:** July 10, 2026

#### Agent de Scoring

In [78]:
# Tools
@tool
def get_client_data_for_scoring(name):
    '''
        Get customer information for scoring between 0 and 100
    '''
    data = {
        "name": "Sophie Martin",
        "company": "Tech Solutions",
        "sector": "IT",
        "employees": 45,
        "revenue": 3200000,
        
        # Purchase behavior
        "last_purchase": "2026-06-28",
        "total_purchases": 12,
        "total_spent": 28450,
        "avg_basket": 2370,
        "purchase_frequency_months": 2.5,
        
        # Engagement
        "email_open_rate": 0.68,
        "click_rate": 0.32,
        "website_visits_30d": 18,
        "time_on_site_min": 6.5,
        "cart_abandonments": 2,
        
        # Risk indicators
        "payment_delay_days": 0,
        "support_tickets": 1,
        "satisfaction_score": 4.5,
        "credit_score": 780,
        "competitor_mentions": 0,
        
        # Contract
        "contract_end": "2027-01-15",
        "has_app": True,
        "subscribed": True
    } 
    return data



In [79]:
# Create agent
sys_prompt = """
        You are a Scroing AI Agent specialized in give personalized score to each client.
        When a user requests a scoring offer:
            1. First Get All sutomer data
            2. Analyze customer profile
            3. give score to this custumer
            4. at end give feedback about this user
"""
scoring_agent = create_agent(
    model= basic_groq_llm,
    system_prompt=sys_prompt,
    tools=[get_client_data_for_scoring],
    debug=True,
)

In [80]:
# Invoke a question
config = {"configurable": {"thread_id": 1}}
resp = scoring_agent.invoke(
    input={"messages": [HumanMessage("give to this client abdellah karani a score ?")]},
    config = config
)
display(Markdown(resp["messages"][-1].content))

[values] {'messages': [HumanMessage(content='give to this client abdellah karani a score ?', additional_kwargs={}, response_metadata={}, id='ac8bfb67-1ef9-4dd9-87c3-5fca32f7be7e')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants a score for Abdellah Karani. First, I need to get his data using the get_client_data_for_scoring function. Let me check if the function is available. Yes, it is. The function requires the name parameter. I\'ll call it with the name "abdellah karani". Once I get the data, I\'ll analyze his profile, look at his history, behavior, and other relevant factors. Then, I\'ll assign a score between 0 and 100. Finally, I\'ll provide feedback based on the score. Make sure the response is clear and helpful.\n', 'tool_calls': [{'id': 'm7cr5sn2f', 'function': {'arguments': '{"name":"abdellah karani"}', 'name': 'get_client_data_for_scoring'}, 'type': 'function'}]}, response_metadata={'token_usage': {'compl

**Score for Sophie Martin (Tech Solutions): 70/100**

**Analysis & Feedback:**  
- **Strengths:**  
  - Strong financial profile (revenue: €3.2M, credit score: 780).  
  - High engagement with emails (68% open rate, 32% click rate).  
  - Timely payments (0 delays) and high satisfaction (4.5/5).  
  - Active app usage and subscription.  

- **Opportunities for Improvement:**  
  - Low time on website (6.5 min/visit) suggests limited engagement depth.  
  - Contract expires in January 2027—proactive renewal outreach is critical.  
  - Only 12 total purchases over 2.5 years: consider upselling/cross-selling.  

**Actionable Recommendations:**  
1. Offer a loyalty incentive or contract extension discount before January 2027.  
2. Personalize email content to increase time spent on the website.  
3. Propose tailored IT solutions to boost purchase frequency.  

Let me know if you need further insights! 🚀

#### Build MultiAgent